# LLM Distillation / Fine-Tuning Demo

This notebook walks through the full pipeline for fine-tuning a small LLM on a
**single-file HTML website generation** task using **Unsloth** + **HuggingFace TRL**.

The workflow is:
1. **Generate training data** — query a powerful model via OpenRouter to get high-quality HTML examples.
2. **Evaluate the base model** — run the small model before fine-tuning on our test cases.
3. **Fine-tune** — apply QLoRA with Unsloth.
4. **Evaluate the fine-tuned model** — compare before / after quality in the browser.

---
**Hardware:** RTX 2080 Ti (22 GB VRAM) — the defaults are tuned for this card.  
**Environment:** run this notebook inside the `distill_dev` Docker container started with:
```bash
sudo docker compose -f docker-compose-distill.yml up
```
Then open `http://localhost:8888` and enter the token `llmdistill_4699`.

## 0. Configuration

In [ ]:
import os

# -----------------------------------------------------------------------
# Set your keys here OR export them as environment variables before
# starting the container.
# -----------------------------------------------------------------------
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "<your-openrouter-key>")
HF_TOKEN          = os.getenv("HF_TOKEN", "")   # optional — only needed to push to Hub

# Teacher model used to generate training data (via OpenRouter)
TEACHER_MODEL = "anthropic/claude-3.5-sonnet"
# Alternatively: "mistralai/mistral-large", "openai/gpt-4o-mini", etc.

# Small student model to fine-tune
STUDENT_MODEL = "unsloth/Llama-3.2-3B-Instruct"
# Alternatives: "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
#               "ibm-granite/granite-3.1-2b-instruct"

# Paths
DATASET_PATH   = "./website_dataset.jsonl"
ADAPTER_DIR    = "./outputs/lora_model"
EVAL_BASE_DIR  = "./eval_results/base"
EVAL_FT_DIR    = "./eval_results/finetuned"

N_SAMPLES      = 200   # number of training examples to generate
EPOCHS         = 3
BATCH_SIZE     = 2
GRAD_ACCUM     = 4     # effective batch = 2*4 = 8

print("Configuration loaded.")

## 1. Generate Training Data

We call a capable teacher model via **OpenRouter** to produce high-quality
single-file HTML examples across a variety of website types.  
Each record is saved as a JSON line: `{instruction, input, output}`.

In [ ]:
# Run the data-generation script (streams output directly)
!python generate_training_data.py \
    --api_key  "{OPENROUTER_API_KEY}" \
    --model    "{TEACHER_MODEL}" \
    --n_samples {N_SAMPLES} \
    --output   "{DATASET_PATH}"

In [ ]:
# Preview a few samples
import json
with open(DATASET_PATH) as fh:
    lines = fh.readlines()
print(f"Total samples: {len(lines)}")
sample = json.loads(lines[0])
print("\n--- Instruction ---")
print(sample["instruction"])
print("\n--- Output (first 500 chars) ---")
print(sample["output"][:500])

## 2. Evaluate the Base Model (Before Fine-Tuning)

We run the student model on 5 test prompts and save the HTML files.  
Open them in a browser to visually inspect quality.

In [ ]:
!python evaluate_model.py \
    --model_name  "{STUDENT_MODEL}" \
    --output_dir  "{EVAL_BASE_DIR}"

In [ ]:
# Display the base-model summary
import json, os
with open(os.path.join(EVAL_BASE_DIR, "summary.json")) as fh:
    base_summary = json.load(fh)
for r in base_summary:
    status = "✓" if (r["starts_with_html"] and r["has_body"]) else "✗"
    print(f"{status} {r['id']:20s}  chars={r['char_count']:5d}")

In [ ]:
# Render one of the generated pages inline
from IPython.display import IFrame
IFrame(src=os.path.join(EVAL_BASE_DIR, "todo_app.html"), width="100%", height=500)

## 3. Fine-Tune with Unsloth

We apply **QLoRA** (4-bit quantised LoRA) using Unsloth's optimised kernels.
Expected training time on an RTX 2080 Ti: ~10–20 minutes for 200 samples × 3 epochs.

In [ ]:
!python train_model.py \
    --model_name   "{STUDENT_MODEL}" \
    --dataset_path "{DATASET_PATH}" \
    --output_dir   "{ADAPTER_DIR}" \
    --epochs       {EPOCHS} \
    --batch_size   {BATCH_SIZE} \
    --grad_accum   {GRAD_ACCUM}

## 4. Evaluate the Fine-Tuned Model

Same 5 test prompts, now using the LoRA adapter.

In [ ]:
!python evaluate_model.py \
    --model_name   "{STUDENT_MODEL}" \
    --adapter_path "{ADAPTER_DIR}" \
    --output_dir   "{EVAL_FT_DIR}"

## 5. Before / After Comparison

In [ ]:
import json, os
with open(os.path.join(EVAL_BASE_DIR, "summary.json")) as fh:
    base_results = {r["id"]: r for r in json.load(fh)}
with open(os.path.join(EVAL_FT_DIR, "summary.json")) as fh:
    ft_results = {r["id"]: r for r in json.load(fh)}

print(f"{'Test Case':<22} {'Base chars':>12} {'FT chars':>10} {'Base ✓':>8} {'FT ✓':>6}")
print("-" * 62)
for tc_id in base_results:
    b = base_results[tc_id]
    f = ft_results.get(tc_id, {})
    b_ok = "✓" if (b.get("starts_with_html") and b.get("has_body")) else "✗"
    f_ok = "✓" if (f.get("starts_with_html") and f.get("has_body")) else "✗"
    print(f"{tc_id:<22} {b['char_count']:>12} {f.get('char_count', 0):>10} {b_ok:>8} {f_ok:>6}")

In [ ]:
# Side-by-side comparison of the todo app
from IPython.display import display, HTML

display(HTML("<h3>Base model — Todo App</h3>"))
display(IFrame(src=os.path.join(EVAL_BASE_DIR, "todo_app.html"), width="100%", height=400))

display(HTML("<h3>Fine-tuned model — Todo App</h3>"))
display(IFrame(src=os.path.join(EVAL_FT_DIR, "todo_app.html"), width="100%", height=400))

## 6. (Optional) Interactive Inference

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=STUDENT_MODEL,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
FastLanguageModel.for_inference(model)
print("Model ready.")

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert web developer. "
    "When asked to build a website, you produce a single self-contained HTML file "
    "that includes all CSS and JavaScript inline. "
    "Output ONLY the raw HTML — no explanations, no markdown fences."
)

def generate(instruction: str, max_new_tokens: int = 2048) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": instruction},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=max_new_tokens, temperature=0.6, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

# Try your own prompt!
html = generate("Create a single-file HTML website: a dark-mode toggle demo page.")
print(html[:800])

In [ ]:
# Render inline
from IPython.display import display, HTML as _HTML
display(_HTML(html))

## 7. (Optional) Push Adapter to HuggingFace Hub

In [ ]:
# Uncomment and fill in your HF repo id to publish the adapter
# HF_REPO = "your-username/llama-3.2-3b-website-lora"
# from huggingface_hub import login
# login(token=HF_TOKEN)
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print(f"Pushed to https://huggingface.co/{HF_REPO}")